In [3]:
import pandas as pd
import json
import random
import os

# 1. DOSSIERS
os.makedirs("data/processed", exist_ok=True)

# 2. CHARGEMENT DE TON LEXIQUE (522 MOTS)
chatbot_data = []
try:
    local_df = pd.read_csv('mon_lexique.csv')
    c1, c2 = local_df.columns[0], local_df.columns[1]
    for _, row in local_df.iterrows():
        chatbot_data.append({
            "instruction": f"Comment dit-on en Dioula : {row[c1]} ?",
            "response": f"En Dioula, on dit : {row[c2]}."
        })
    print(f"✅ Lexique local : {len(chatbot_data)} mots ajoutés.")
except Exception as e:
    print(f"❌ Erreur locale : {e}")

# 3. CHARGEMENT MASAKHANE (Format Parquet - Le plus fiable)
print("--- Tentative Masakhane via Parquet ---")
try:
    # URL vers le stockage Parquet de Hugging Face pour fra-bam
    url_parquet = "https://huggingface.co/datasets/masakhane/mafand/resolve/main/data/fra-bam/train.parquet"
    df_m = pd.read_parquet(url_parquet)

    count_m = 0
    for _, row in df_m.iterrows():
        # On extrait selon la structure Parquet (souvent une colonne 'translation' contenant un dict)
        if 'translation' in row:
            f = row['translation']['fra']
            b = row['translation']['bam']
        else:
            f, b = row['source_sentence'], row['target_sentence']

        chatbot_data.append({
            "instruction": f"Comment dit-on en Dioula : {f} ?",
            "response": f"En Dioula, on dit : {b}."
        })
        count_m += 1
    print(f"✅ Masakhane : {count_m} phrases ajoutées.")
except Exception as e:
    print(f"⚠️ Échec Parquet : {e}. On tente le CSV brut...")
    try:
        url_csv = "https://huggingface.co/datasets/masakhane/mafand/resolve/main/data/fra-bam/train.csv"
        df_c = pd.read_csv(url_csv)
        for _, row in df_c.iterrows():
            chatbot_data.append({"instruction": f"Comment dit-on en Dioula : {row[0]} ?", "response": f"En Dioula, on dit : {row[1]}."})
        print(f"✅ Masakhane (via CSV) ajouté !")
    except:
        print("❌ Masakhane indisponible. On reste sur tes 522 mots.")

# 4. SAUVEGARDE
if chatbot_data:
    random.shuffle(chatbot_data)
    with open("data/processed/final_chatbot_data.jsonl", "w", encoding="utf-8") as f:
        for entry in chatbot_data:
            json.dump(entry, f, ensure_ascii=False)
            f.write("\n")
    print(f"\n🚀 TOTAL FINAL : {len(chatbot_data)} lignes prêtes !")

❌ Erreur locale : [Errno 2] No such file or directory: 'mon_lexique.csv'
--- Tentative Masakhane via Parquet ---
⚠️ Échec Parquet : HTTP Error 404: Not Found. On tente le CSV brut...
❌ Masakhane indisponible. On reste sur tes 522 mots.


In [5]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# 1. Initialisation du Tokenizer
model_id = "google/flan-t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_id)

# 2. Fonction de prétraitement (Correction de l'indentation)
def preprocess_function(examples):
    # Tout ce qui est ici doit être décalé d'un cran (4 espaces)
    model_inputs = tokenizer(
        examples["instruction"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["response"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 3. Application au dataset (On s'assure que 'dataset' existe encore)
try:
    tokenized_dataset = dataset.map(preprocess_function, batched=True)
    print("🚀 Dataset prêt pour l'entraînement !")
except NameError:
    print("❌ Erreur : Le 'dataset' a disparu de la mémoire. Relance la cellule précédente qui crée le fichier JSONL.")

Map:   0%|          | 0/522 [00:00<?, ? examples/s]

🚀 Dataset prêt pour l'entraînement !
